# Kural-TTS · Phase 1 baselines

Runs each baseline TTS model over `eval/sentences/feasibility_v1.tsv`, measures speed and memory, then scores the outputs with ASR.

* **Runtime:** GPU. A T4 is enough for every model except svara (3B), which needs an L4 or A100 for reasonable speed.
* **Secrets:** add `HF_TOKEN` in Colab's 🔑 panel. Before running, accept the terms on the gated repos `ai4bharat/indic-parler-tts` and `ai4bharat/IndicF5`.
* Each model gets its **own venv**, because their dependency pins conflict. The venvs reuse Colab's preinstalled torch.
* Results are copied to Google Drive at the end.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [ ]:
# Get the code: either clone (after you push the repo) or copy from Drive.
REPO_URL = "https://github.com/Mamlesh18/kural-tts.git"  # edit if different
!git clone -q {REPO_URL} /content/kural-tts || echo "clone failed: upload the repo or mount Drive"
%cd /content/kural-tts
!pip -q install uv

In [ ]:
import subprocess


def run_model(name, device="cuda", limit=0, extra_env=None):
    env_dir = f"/content/envs/{name}"
    if not os.path.exists(env_dir):
        subprocess.run(f"uv venv -q --system-site-packages {env_dir}", shell=True, check=True)
        subprocess.run(
            f"uv pip install -q --python {env_dir}/bin/python -r scripts/baseline/envs/{name}.txt",
            shell=True,
            check=True,
        )
    cmd = f"{env_dir}/bin/python scripts/baseline/run_baseline.py --model {name} --device {device}"
    if limit:
        cmd += f" --limit {limit}"
    env = {**os.environ, **(extra_env or {})}
    subprocess.run(cmd, shell=True, check=False, env=env)

## 1 · GPU runs

In [ ]:
run_model("mms_tam")

In [ ]:
run_model("indic_parler")

In [ ]:
run_model("coqui_vits_en")

**IndicF5** clones a voice from a reference clip. Upload a 6–10 s **Tamil** clip that you have the rights to use, plus its exact transcript.

In [ ]:
run_model(
    "indic_f5",
    extra_env={
        "F5_REF_AUDIO": "/content/ref_ta.wav",
        "F5_REF_TEXT": "<exact Tamil transcript of the reference clip>",
    },
)

**svara-TTS (experimental decoding).** Listen to a few outputs first. If they're noise, the token layout differs from Orpheus, so check `Kenpath/svara-tts-inference`.

In [ ]:
run_model("svara", extra_env={"SVARA_VOICE": "Tamil (Female)"})

## 2 · CPU speed (5 sentences per category)
This matters because Kural-TTS ships as a local pip package.

In [ ]:
for m in ["mms_tam", "coqui_vits_en", "indic_parler"]:
    run_model(m, device="cpu", limit=5)

## 3 · Scoring (ASR CER/WER + UTMOS)

In [ ]:
run_model_env = "/content/envs/scoring"
if not os.path.exists(run_model_env):
    !uv venv -q --system-site-packages {run_model_env}
    !uv pip install -q --python {run_model_env}/bin/python -r scripts/baseline/envs/scoring.txt
!{run_model_env}/bin/python scripts/baseline/score.py --utmos
!cat reports/feasibility_scores.md

## 4 · Listening test + save to Drive

In [ ]:
!python scripts/baseline/make_listening_test.py
from google.colab import drive

drive.mount("/content/drive")
DEST = "/content/drive/MyDrive/kural-tts/phase1"
!mkdir -p {DEST}
!cp -r outputs reports {DEST}/
print("Saved to", DEST)